# EXT — Knowledge-Enhanced Model

**External knowledge, two ways:**
1. **Multilingual backbone (MuRIL)** — pretrained on 17 Indian languages incl. Bengali + transliteration, so it brings external cross-lingual knowledge. (Swap to `csebuetnlp/banglabert` via `CFG.base_name` if you prefer.)
2. **Lexicon gazetteer** — an external toxic/identity/threat/sexual/profanity word list injected as features (counts per category + romanization/punctuation signals). Drop a file `lexicon.txt` (one `category<TAB>word` per line) into the input and it is used automatically; otherwise a small seed lexicon is used.

The fine-tuned transformer is blended with a classic head over **[TF-IDF ⊕ lexicon features]**, metric-tuned + gated. **GPU + Internet ON, Run All.**


In [ ]:
# ---- 1. Install (Internet ON) ----
import sys, subprocess
def _pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=False)
_pip("iterative-stratification==0.1.7")
_pip("git+https://github.com/csebuetnlp/normalizer")
print("deps done")

In [ ]:
# ---- 2. Imports & CONFIG ----
import os, re, gc, glob, random, warnings, math
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")
from transformers import (AutoTokenizer, AutoModel, AutoConfig,
                          get_linear_schedule_with_warmup, logging as hf_logging)
hf_logging.set_verbosity_error()
class CFG:
    model_name  = "google/muril-base-cased"     # multilingual external knowledge
    base_name   = "csebuetnlp/banglabert"
    max_len     = 160
    n_folds     = 5
    epochs      = 4
    batch_size  = 16
    grad_accum  = 1
    eval_bs     = 32
    lr          = 2e-5
    weight_decay= 0.01
    warmup_ratio= 0.06
    max_grad_norm=1.0
    dropout     = 0.3
    lstm_hidden = 256
    focal_gamma = 2.0
    pos_weight_clip = 20.0
    aug_copies  = 2
    GATE        = True
    seed        = 42
    use_fp16    = True
LABELS=["is_toxic","implicit_toxicity","sarcasm","contains_profanity","threatening","identity_attack","sexual_content"]
RARE=["implicit_toxicity","sarcasm","threatening","identity_attack","sexual_content"]
N_LABELS=len(LABELS)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG.seed)
print("Device:",DEVICE,"| Model:",CFG.model_name)

In [ ]:
# ---- 3. Metric helpers (Combined=(A+B)/2) ----
def gate_preds(prob,thr,gate=True):
    pred=(prob>=thr).astype(int)
    if gate: pred[pred[:,0]==0,1:]=0
    return pred
def combined_score(y,pred):
    A=f1_score(y[:,0],pred[:,0],average="macro",zero_division=0)
    B=np.mean([f1_score(y[:,i],pred[:,i],zero_division=0) for i in range(1,N_LABELS)])
    return (A+B)/2.0,A,B
def tune_thresholds_metric(y,prob):
    grid=np.arange(0.05,0.991,0.01); thr=np.full(N_LABELS,0.5)
    bt,bs=0.5,-1
    for t in grid:
        s=f1_score(y[:,0],(prob[:,0]>=t).astype(int),average="macro",zero_division=0)
        if s>bs: bs,bt=s,t
    thr[0]=bt
    for i in range(1,N_LABELS):
        bt,bs=0.5,-1
        for t in grid:
            s=f1_score(y[:,i],(prob[:,i]>=t).astype(int),zero_division=0)
            if s>bs: bs,bt=s,t
        thr[i]=bt
    return thr

In [ ]:
# ---- 4. Load data ----
def find_data_dir():
    # 1) Prefer the OFFICIAL competition mount (its test IDs are what Kaggle scores)
    for c in ["/kaggle/input/competitions/bengali-nlp-safety-challenge-2026","/kaggle/input/bengali-nlp-safety-challenge-2026"]:
        if os.path.exists(os.path.join(c,"train.csv")) and os.path.exists(os.path.join(c,"sample_submission.csv")):
            return c
    # 2) A folder that has BOTH train and sample_submission (a complete competition copy)
    for tp in sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True), key=len):
        d=os.path.dirname(tp)
        if os.path.exists(os.path.join(d,"sample_submission.csv")): return d
    # 3) Fallbacks
    for c in ["./data","."]:
        if os.path.exists(os.path.join(c,"train.csv")): return c
    print("Contents of /kaggle/input:")
    for r,_,fs in os.walk("/kaggle/input"):
        for f in fs: print("   ",os.path.join(r,f))
    raise FileNotFoundError("Attach the competition dataset as input.")
DATA_DIR=find_data_dir(); print("Data dir:",DATA_DIR)
train_df=pd.read_csv(os.path.join(DATA_DIR,"train.csv")); test_df=pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
try: val_df=pd.read_csv(os.path.join(DATA_DIR,"validation.csv"))
except FileNotFoundError: val_df=None
try: sample_sub=pd.read_csv(os.path.join(DATA_DIR,"sample_submission.csv"))
except FileNotFoundError: sample_sub=None
def pick_col(df,prefer,kw):
    for p in prefer:
        if p in df.columns: return p
    for c in df.columns:
        if any(k in c.lower() for k in kw): return c
    o=[c for c in df.columns if df[c].dtype==object]
    return max(o,key=lambda c: df[c].astype(str).str.len().mean())
TEXT_COL=pick_col(train_df,["text"],["text","comment","sentence"]); ID_COL=pick_col(test_df,["ID","id","Id"],["id"])
if val_df is not None and set(LABELS).issubset(val_df.columns):
    full_df=pd.concat([train_df,val_df],ignore_index=True); print("train+validation")
else:
    full_df=train_df.copy(); print("train only")
full_df=full_df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
for l in LABELS: full_df[l]=full_df[l].astype(int)
print("Full",full_df.shape,"Test",test_df.shape)

In [ ]:
# ---- 5. Cleaning + EDA augmentation ----
try:
    from normalizer import normalize as bn_normalize; _HAS_NORM=True; print("normalizer ON")
except Exception as e:
    _HAS_NORM=False; print("basic cleaning:",e)
URL_RE=re.compile(r"http\S+|www\.\S+"); HTML_RE=re.compile(r"<.*?>")
SPACE_RE=re.compile(r"\s+"); KEEP_RE=re.compile(r"[^\u0980-\u09FF\u0020-\u007E]"); ELONG_RE=re.compile(r"(.)\1{2,}")
def clean_text(t):
    t=str(t); t=HTML_RE.sub(" ",t); t=URL_RE.sub(" ",t)
    t=t.replace("\u200c"," ").replace("\u200d"," "); t=KEEP_RE.sub(" ",t); t=ELONG_RE.sub(r"\1\1",t)
    t=SPACE_RE.sub(" ",t).strip()
    if _HAS_NORM:
        try: t=bn_normalize(t)
        except Exception: pass
    return t if t else "[EMPTY]"
full_df["clean"]=full_df[TEXT_COL].apply(clean_text); test_df["clean"]=test_df[TEXT_COL].apply(clean_text)
# ---- Deduplicate identical texts via majority-vote labels (removes redundancy + annotation noise) ----
DEDUP=True
if DEDUP:
    _before=len(full_df)
    _agg=full_df.groupby("clean",as_index=False)[LABELS].mean()
    for _l in LABELS: _agg[_l]=(_agg[_l]>=0.5).astype(int)
    _first=full_df.drop_duplicates("clean")[["clean",TEXT_COL]+([ "ID"] if "ID" in full_df.columns else [])]
    full_df=_first.merge(_agg,on="clean").reset_index(drop=True)
    print(f"dedup: {_before} -> {len(full_df)} rows (majority-vote labels)")
def eda(text,p_del=0.1):
    w=text.split()
    if len(w)<4: return text
    w=[x for x in w if random.random()>p_del] or w
    if len(w)>=2:
        i,j=random.sample(range(len(w)),2); w[i],w[j]=w[j],w[i]
    return " ".join(w)
def augment_rows(sub_df):
    et,el=[],[]
    if CFG.aug_copies<=0: return et,el
    rr=sub_df[sub_df[RARE].sum(axis=1)>0]
    for _ in range(CFG.aug_copies):
        for _,r in rr.iterrows():
            et.append(eda(r["clean"])); el.append(r[LABELS].values.astype(float))
    return et,el
print(full_df[["clean"]].head(2).to_string())

In [ ]:
# ---- 6. External lexicon (knowledge) features ----
# Seed lexicon (extend by adding /kaggle/input/**/lexicon.txt with lines: category<TAB>word)
SEED_LEXICON = {
    "profanity": ["\u09b6\u09be\u09b2\u09be","\u0996\u09be\u09a8\u0995\u09bf","\u099a\u09c1\u09a4\u09bf\u09af\u09bc\u09be","\u09ac\u09be\u09b2","\u0995\u09c1\u09a4\u09cd\u09a4\u09be","\u09b9\u09be\u09b0\u09be\u09ae\u09bf","fuck","chutia","magi","kutta"],
    "identity":  ["\u09ae\u09be\u09b2\u09be\u0989\u09a8","\u09b9\u09bf\u09a8\u09cd\u09a6\u09c1","\u09ae\u09c1\u09b8\u09b2\u09ae\u09be\u09a8","\u0995\u09be\u09ab\u09c7\u09b0","\u09ae\u09cc\u09b2\u09ac\u09be\u09a6\u09c0","malaun","kafer"],
    "threat":    ["\u09ae\u09be\u09b0\u09ac","\u0995\u09c1\u09aa\u09bf\u09af\u09bc\u09c7","\u0996\u09c1\u09a8","\u099c\u09ac\u09be\u0987","\u09b6\u09c7\u09b7","kill","\u09ae\u09c7\u09b0\u09c7 \u09ab\u09c7\u09b2"],
    "sexual":    ["\u09b8\u09c7\u0995\u09cd\u09b8","\u09a7\u09b0\u09cd\u09b7\u09a3","\u09a8\u0997\u09cd\u09a8","\u099a\u09c1\u09a6","sex","porn","rape"],
}
def load_lexicon():
    lex={k:set(v) for k,v in SEED_LEXICON.items()}
    for path in glob.glob("/kaggle/input/**/lexicon.txt", recursive=True):
        try:
            for line in open(path,encoding="utf-8"):
                if "\t" in line:
                    c,w=line.rstrip("\n").split("\t",1); lex.setdefault(c,set()).add(w.strip())
            print("loaded external lexicon:",path)
        except Exception as e: print("lexicon read fail",e)
    return {k:sorted(v) for k,v in lex.items()}
LEX=load_lexicon(); LEX_CATS=list(LEX.keys())
def lexicon_features(texts):
    F=np.zeros((len(texts), len(LEX_CATS)+2), dtype=np.float32)
    for i,t in enumerate(texts):
        s=str(t)
        for j,c in enumerate(LEX_CATS):
            F[i,j]=sum(s.count(w) for w in LEX[c])
        F[i,-2]=sum(ch.isascii() and ch.isalpha() for ch in s)   # romanized letters
        F[i,-1]=s.count("!")
    return F
print("lexicon categories:",LEX_CATS,"| sizes:",{k:len(v) for k,v in LEX.items()})

In [ ]:
# ---- 7. Tokenizer, Dataset, hybrid model (fp16-safe), Focal loss ----
try:
    tokenizer=AutoTokenizer.from_pretrained(CFG.model_name); MODEL_NAME=CFG.model_name
except Exception as e:
    print("backbone failed, base:",e); tokenizer=AutoTokenizer.from_pretrained(CFG.base_name); MODEL_NAME=CFG.base_name
print("Backbone:",MODEL_NAME)
class ToxDS(Dataset):
    def __init__(self,texts,labels=None): self.texts=list(texts); self.labels=labels
    def __len__(self): return len(self.texts)
    def __getitem__(self,i):
        enc=tokenizer(self.texts[i],truncation=True,max_length=CFG.max_len,padding="max_length",return_tensors="pt")
        it={k:v.squeeze(0) for k,v in enc.items()}
        if self.labels is not None: it["labels"]=torch.tensor(self.labels[i],dtype=torch.float)
        return it
class HybridModel(nn.Module):
    def __init__(self,model_name,n_out):
        super().__init__()
        self.config=AutoConfig.from_pretrained(model_name); self.backbone=AutoModel.from_pretrained(model_name)
        h=self.config.hidden_size; L=CFG.lstm_hidden
        self.lstm=nn.LSTM(h,L,batch_first=True,bidirectional=True)
        self.attn=nn.Linear(2*L,1); self.drop=nn.Dropout(CFG.dropout)
        self.head=nn.Sequential(nn.Linear(4*L,2*L),nn.GELU(),nn.Dropout(CFG.dropout),nn.Linear(2*L,n_out))
    def forward(self,input_ids,attention_mask,token_type_ids=None):
        kw=dict(input_ids=input_ids,attention_mask=attention_mask)
        if token_type_ids is not None: kw["token_type_ids"]=token_type_ids
        seq=self.backbone(**kw).last_hidden_state; lo,_=self.lstm(seq); m=attention_mask.unsqueeze(-1)
        neg=torch.finfo(lo.dtype).min
        sc=self.attn(lo).masked_fill(m==0,neg); w=torch.softmax(sc,dim=1)
        ap=(w*lo).sum(1); mp=lo.masked_fill(m==0,neg).max(1).values
        return self.head(self.drop(torch.cat([ap,mp],-1)))
class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,pos_weight=None): super().__init__(); self.gamma=gamma; self.pos_weight=pos_weight
    def forward(self,logits,targets):
        bce=F.binary_cross_entropy_with_logits(logits,targets,reduction="none",pos_weight=self.pos_weight)
        p=torch.sigmoid(logits); pt=p*targets+(1-p)*(1-targets)
        return (((1-pt)**self.gamma)*bce).mean()
def pos_weight_from(y):
    pos=y.sum(0); neg=len(y)-pos
    return np.clip(neg/np.clip(pos,1,None),1.0,CFG.pos_weight_clip)
@torch.no_grad()
def predict_probs(model,loader):
    model.eval(); out=[]
    for b in loader:
        ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE)
        tti=b.get("token_type_ids"); tti=tti.to(DEVICE) if tti is not None else None
        with torch.cuda.amp.autocast(enabled=CFG.use_fp16): logit=model(ids,mask,tti)
        out.append(torch.sigmoid(logit).float().cpu().numpy())
    return np.concatenate(out,0)
def train_transformer_fold(tr_texts,tr_y,va_texts,va_y,test_texts,fold):
    seed_everything(CFG.seed+fold); model=HybridModel(MODEL_NAME,N_LABELS).to(DEVICE)
    trl=DataLoader(ToxDS(tr_texts,tr_y),batch_size=CFG.batch_size,shuffle=True,num_workers=0,pin_memory=True)
    val=DataLoader(ToxDS(va_texts,va_y),batch_size=CFG.eval_bs,shuffle=False,num_workers=0)
    tel=DataLoader(ToxDS(test_texts),batch_size=CFG.eval_bs,shuffle=False,num_workers=0)
    pw=torch.tensor(pos_weight_from(tr_y),dtype=torch.float,device=DEVICE); crit=FocalLoss(CFG.focal_gamma,pw)
    nd=["bias","LayerNorm.weight"]
    params=[{"params":[p for n,p in model.named_parameters() if not any(x in n for x in nd)],"weight_decay":CFG.weight_decay},
            {"params":[p for n,p in model.named_parameters() if any(x in n for x in nd)],"weight_decay":0.0}]
    opt=torch.optim.AdamW(params,lr=CFG.lr); steps=max(1,(len(trl)//CFG.grad_accum))*CFG.epochs
    sch=get_linear_schedule_with_warmup(opt,int(steps*CFG.warmup_ratio),steps)
    scaler=torch.cuda.amp.GradScaler(enabled=CFG.use_fp16); best,bv,bt=-1,None,None
    for ep in range(CFG.epochs):
        model.train(); run=0.0; opt.zero_grad()
        for st,b in enumerate(trl):
            ids=b["input_ids"].to(DEVICE); mask=b["attention_mask"].to(DEVICE)
            tti=b.get("token_type_ids"); tti=tti.to(DEVICE) if tti is not None else None; y=b["labels"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=CFG.use_fp16): loss=crit(model(ids,mask,tti),y)/CFG.grad_accum
            scaler.scale(loss).backward(); run+=loss.item()*CFG.grad_accum
            if (st+1)%CFG.grad_accum==0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),CFG.max_grad_norm)
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad()
        vp=predict_probs(model,val); th=tune_thresholds_metric(va_y,vp)
        c,A,Bv=combined_score(va_y,gate_preds(vp,th,CFG.GATE))
        print(f"  fold{fold} ep{ep+1}/{CFG.epochs} loss={run/len(trl):.4f} valCombined={c:.4f}")
        if c>=best: best=c; bv=vp; bt=predict_probs(model,tel)
    del model; gc.collect(); torch.cuda.empty_cache()
    return bv,bt
print("transformer ready")

In [ ]:
# ---- 8. Classic head over [TF-IDF + lexicon features] ----
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
def train_classic_fold(tr_texts,tr_y,va_texts,test_texts):
    wv=TfidfVectorizer(analyzer="word",ngram_range=(1,2),min_df=2,max_features=60000,sublinear_tf=True)
    cv=TfidfVectorizer(analyzer="char_wb",ngram_range=(2,5),min_df=2,max_features=60000,sublinear_tf=True)
    Tt=hstack([wv.fit_transform(tr_texts),cv.fit_transform(tr_texts)]).tocsr()
    Tv=hstack([wv.transform(va_texts),cv.transform(va_texts)]).tocsr()
    Te=hstack([wv.transform(test_texts),cv.transform(test_texts)]).tocsr()
    Ltr=lexicon_features(tr_texts); Lva=lexicon_features(va_texts); Lte=lexicon_features(test_texts)
    sc=StandardScaler().fit(Ltr)
    Ftr=hstack([Tt,csr_matrix(sc.transform(Ltr))]).tocsr()
    Fva=hstack([Tv,csr_matrix(sc.transform(Lva))]).tocsr()
    Fte=hstack([Te,csr_matrix(sc.transform(Lte))]).tocsr()
    vp=np.zeros((Fva.shape[0],N_LABELS)); ep=np.zeros((Fte.shape[0],N_LABELS))
    for i in range(N_LABELS):
        if tr_y[:,i].sum()==0: continue
        clf=LogisticRegression(C=3.0,class_weight="balanced",max_iter=2000).fit(Ftr,tr_y[:,i].astype(int))
        vp[:,i]=clf.predict_proba(Fva)[:,1]; ep[:,i]=clf.predict_proba(Fte)[:,1]
    return vp,ep
print("classic+lexicon ready")

In [ ]:
# ---- 9. CV ----
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
    folds=list(MultilabelStratifiedKFold(n_splits=CFG.n_folds,shuffle=True,random_state=CFG.seed).split(full_df["clean"].values,full_df[LABELS].values)); print("MLStratKFold")
except Exception as e:
    from sklearn.model_selection import StratifiedKFold
    folds=list(StratifiedKFold(n_splits=CFG.n_folds,shuffle=True,random_state=CFG.seed).split(full_df["clean"].values,full_df["is_toxic"].values)); print("StratKFold",e)
X=full_df["clean"].values; Y=full_df[LABELS].values.astype(float); test_texts=list(test_df["clean"].values)
oof_tf=np.zeros((len(full_df),N_LABELS)); oof_clf=np.zeros((len(full_df),N_LABELS))
test_tf=np.zeros((len(test_df),N_LABELS)); test_clf=np.zeros((len(test_df),N_LABELS))
for fold,(tr,va) in enumerate(folds):
    print(f"===== Fold {fold+1}/{CFG.n_folds} =====")
    sub_tr=full_df.iloc[tr]; tr_t=list(X[tr]); tr_y=Y[tr].copy()
    et,el=augment_rows(sub_tr)
    if et: tr_t+=et; tr_y=np.vstack([tr_y,np.array(el)]); print(f"  +{len(et)} augmented")
    vtf,ttf=train_transformer_fold(np.array(tr_t,dtype=object),tr_y,X[va],Y[va],test_texts,fold)
    oof_tf[va]=vtf; test_tf+=ttf/CFG.n_folds
    vclf,tclf=train_classic_fold(list(X[tr]),Y[tr],list(X[va]),test_texts)
    oof_clf[va]=vclf; test_clf+=tclf/CFG.n_folds
print("CV complete.")

In [ ]:
# ---- 10. Blend + thresholds + report ----
best=(-1,None,None)
for w in np.round(np.arange(0.0,1.001,0.05),2):
    bl=w*oof_tf+(1-w)*oof_clf; th=tune_thresholds_metric(Y,bl)
    c,_,_=combined_score(Y,gate_preds(bl,th,CFG.GATE))
    if c>best[0]: best=(c,w,th)
_,best_w,thr=best
oof_blend=best_w*oof_tf+(1-best_w)*oof_clf; oof_pred=gate_preds(oof_blend,thr,CFG.GATE)
c,A,B=combined_score(Y,oof_pred)
print("="*54); print(f"Blend transformer={best_w} classic(tfidf+lexicon)={round(1-best_w,2)}")
print(f"OOF Subtask A={A:.4f} | Subtask B={B:.4f}")
print(f"OOF COMBINED SCORE = {c:.4f}   <-- expected leaderboard"); print("="*54)
for i,l in enumerate(LABELS):
    ff=f1_score(Y[:,i],oof_pred[:,i],average=("macro" if i==0 else "binary"),zero_division=0)
    print(f"  {l:<20} thr={thr[i]:.2f} F1={ff:.4f}")

In [ ]:
# ---- Build submission (official test.csv IDs — proven format) ----
test_blend=best_w*test_tf+(1-best_w)*test_clf
final=gate_preds(test_blend, thr, CFG.GATE)
sub=pd.DataFrame({ID_COL:test_df[ID_COL].values})
for i,l in enumerate(LABELS): sub[l]=final[:,i]
sub.to_csv("submission.csv", index=False)
print("submission.csv", sub.shape, "| IDs taken from official test.csv")
print(sub.head())
